# KLA PS01 -- Restoration Model Training (Colab)

Trains the NAFNet-lite denoise+2x-SR model on KLA's paired semiconductor
inspection dataset. Built and pipeline-tested locally on CPU with real
data before this notebook was written -- this just runs the same code on
a real GPU.

**Before running:** Runtime -> Change runtime type -> **T4 GPU** (or better,
if you have Colab Pro / A100 access).

**You need to upload the dataset first.** Zip your local `train/` folder
(containing `GT/` and `NoisyLR/` subfolders of `.npy` files) and upload it
to your Google Drive, e.g. as `MyDrive/kla_ps01/train.zip`. Update the path
in the "Locate dataset" cell below to match.


## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Locate / unzip dataset

Edit `DRIVE_ZIP_PATH` to point at wherever you uploaded `train.zip`
(the folder must contain `GT/` and `NoisyLR/` subfolders of matching
`.npy` files -- same format as the KLA delivery).

In [ ]:
import os, zipfile, shutil

DRIVE_ZIP_PATH = "/content/drive/MyDrive/kla_ps01/train.zip"  # <-- EDIT THIS
EXTRACT_ROOT = "/content/data"

assert os.path.isfile(DRIVE_ZIP_PATH), (
    f"Zip not found at {DRIVE_ZIP_PATH} -- open the Drive file browser on the left "
    f"and confirm the exact path/filename, then fix DRIVE_ZIP_PATH above."
)

if os.path.isdir(EXTRACT_ROOT):
    shutil.rmtree(EXTRACT_ROOT)
os.makedirs(EXTRACT_ROOT, exist_ok=True)

# Manual extraction, not zf.extractall(): some Windows zip tools store entries
# using a backslash as the path separator, which is not valid per the ZIP spec.
# On Colab's Linux backend, zipfile.extractall() then treats that as one flat
# filename containing a backslash character instead of a nested folder path.
# Normalizing separators here makes extraction work regardless of which tool
# produced the zip.
BACKSLASH = chr(92)
with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
    for info in zf.infolist():
        norm_name = info.filename.replace(BACKSLASH, "/")
        if norm_name.endswith("/"):
            continue
        dest_path = os.path.join(EXTRACT_ROOT, *norm_name.split("/"))
        os.makedirs(os.path.dirname(dest_path), exist_ok=True)
        with zf.open(info) as src, open(dest_path, "wb") as dst:
            shutil.copyfileobj(src, dst)

DATA_ROOT = None
for root, dirs, files in os.walk(EXTRACT_ROOT):
    if "GT" in dirs and "NoisyLR" in dirs:
        DATA_ROOT = root
        break

if DATA_ROOT is None:
    print("Could not find GT/ and NoisyLR/ folders anywhere under the extracted zip.")
    print("Here's what was actually extracted, so you can see the real structure:")
    for root, dirs, files in os.walk(EXTRACT_ROOT):
        depth = root.replace(EXTRACT_ROOT, "").count(os.sep)
        print("  " * depth + os.path.basename(root) + "/")
        for f in sorted(files)[:3]:
            print("  " * (depth + 1) + f)
    raise FileNotFoundError("GT/NoisyLR not found -- see the directory listing above.")

print("Using DATA_ROOT =", DATA_ROOT)
print("GT files:", len(os.listdir(os.path.join(DATA_ROOT, "GT"))))
print("NoisyLR files:", len(os.listdir(os.path.join(DATA_ROOT, "NoisyLR"))))


## 4. Install dependencies

In [ ]:
!pip install -q torch torchvision numpy pillow lpips scikit-image


## 5. Write source files

Same code that was already built and pipeline-tested locally (CPU smoke
test, real data, verified shapes/losses/outputs) -- reproduced here
verbatim so training runs the exact same model/loss/data logic.

In [ ]:
import os
os.makedirs("/content/src", exist_ok=True)


In [ ]:
%%writefile /content/src/dataset.py
"""
Dataset loader for the KLA semiconductor-inspection restoration task.

Pairs on disk (confirmed by inspecting the actual delivered dataset):
    train/GT/<id>.npy        -> float32, HxW = 256x256, range ~[0, 1]
    train/NoisyLR/<id>.npy   -> float32, HxW = 128x128, range can exceed
                                 [0, 1] (speckle noise pushes pixels past
                                 the true signal range -- this is expected,
                                 per the problem statement, so we do NOT
                                 clip the input before feeding the model).

The held-out folder at the repo root (`NoisyLR/`, no matching GT) mirrors
KLA's blind test set: same 128x128 unclipped format, no ground truth.
"""

import glob
import os
import random

import numpy as np
import torch
from torch.utils.data import Dataset


class RestorationDataset(Dataset):
    def __init__(self, gt_dir, noisy_dir, train=True, extra_noise_std=0.03, crop_size=None):
        self.gt_paths = sorted(glob.glob(os.path.join(gt_dir, "*.npy")))
        self.noisy_dir = noisy_dir
        self.train = train
        self.extra_noise_std = extra_noise_std
        self.crop_size = crop_size

        ids = [os.path.basename(p) for p in self.gt_paths]
        missing = [i for i in ids if not os.path.exists(os.path.join(noisy_dir, i))]
        if missing:
            raise FileNotFoundError(f"{len(missing)} GT files have no matching NoisyLR file, e.g. {missing[:3]}")

    def __len__(self):
        return len(self.gt_paths)

    def _augment_extra_degradation(self, noisy_lr):
        """Domain-randomization on top of the provided degradation, so the
        model doesn't just memorize KLA's fixed noise recipe -- this is
        what the OOD generalization requirement in the problem statement
        is actually testing for."""
        if not self.train:
            return noisy_lr

        # Randomly strengthen speckle noise (multiplicative) at varying levels.
        if random.random() < 0.5:
            speckle_std = random.uniform(0.0, self.extra_noise_std)
            noisy_lr = noisy_lr * (1.0 + np.random.randn(*noisy_lr.shape).astype(np.float32) * speckle_std)

        # Randomly add extra Gaussian noise (additive) at varying levels.
        if random.random() < 0.5:
            gauss_std = random.uniform(0.0, self.extra_noise_std)
            noisy_lr = noisy_lr + np.random.randn(*noisy_lr.shape).astype(np.float32) * gauss_std

        return noisy_lr

    def _geometric_augment(self, gt, noisy_lr):
        if not self.train:
            return gt, noisy_lr

        if random.random() < 0.5:
            gt, noisy_lr = np.fliplr(gt).copy(), np.fliplr(noisy_lr).copy()
        if random.random() < 0.5:
            gt, noisy_lr = np.flipud(gt).copy(), np.flipud(noisy_lr).copy()
        k = random.randint(0, 3)
        if k:
            gt, noisy_lr = np.rot90(gt, k).copy(), np.rot90(noisy_lr, k).copy()

        return gt, noisy_lr

    def __getitem__(self, idx):
        gt_path = self.gt_paths[idx]
        fname = os.path.basename(gt_path)
        noisy_path = os.path.join(self.noisy_dir, fname)

        gt = np.load(gt_path).astype(np.float32)
        noisy_lr = np.load(noisy_path).astype(np.float32)

        gt, noisy_lr = self._geometric_augment(gt, noisy_lr)
        noisy_lr = self._augment_extra_degradation(noisy_lr)

        gt_t = torch.from_numpy(gt).unsqueeze(0)          # 1 x 256 x 256
        noisy_t = torch.from_numpy(noisy_lr).unsqueeze(0)  # 1 x 128 x 128

        return {"noisy_lr": noisy_t, "gt": gt_t, "filename": fname}


def make_splits(gt_dir, noisy_dir, val_fraction=0.1, seed=42):
    """Deterministic train/val split by filename, so the val set stays
    fixed across runs and doesn't leak into training."""
    all_ids = sorted(os.path.basename(p) for p in glob.glob(os.path.join(gt_dir, "*.npy")))
    rng = random.Random(seed)
    shuffled = all_ids[:]
    rng.shuffle(shuffled)
    n_val = max(1, int(len(shuffled) * val_fraction))
    val_ids = set(shuffled[:n_val])
    train_ids = [i for i in all_ids if i not in val_ids]
    val_ids = [i for i in all_ids if i in val_ids]
    return train_ids, val_ids


class SubsetRestorationDataset(RestorationDataset):
    """Same as RestorationDataset but restricted to a given filename list."""

    def __init__(self, gt_dir, noisy_dir, filenames, train=True, extra_noise_std=0.03):
        super().__init__(gt_dir, noisy_dir, train=train, extra_noise_std=extra_noise_std)
        keep = set(filenames)
        self.gt_paths = [p for p in self.gt_paths if os.path.basename(p) in keep]


In [ ]:
%%writefile /content/src/model.py
"""
NAFNet-lite: a compact NAFNet-style encoder-decoder (Chen et al., 2022,
"Simple Baselines for Image Restoration") adapted for this task:
  - single-channel (grayscale) in/out
  - input at 128x128, output at 256x256 -> a PixelShuffle x2 head is
    fused onto the end of the restoration decoder, so denoising and
    super-resolution happen in one pass instead of two separate stages.

Chosen over SwinIR/Restormer for this hackathon because NAFNet gets
competitive-to-SOTA denoising/deblurring quality at much lower compute
(no self-attention, no nonlinear activations) -- and the KLA benchmark
explicitly penalizes slow inference on the H100 grading run.

No adversarial (GAN) training is used anywhere in this project: the
problem statement explicitly warns against "artificial patterns or
ringing", which is the classic GAN-restoration failure mode.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class LayerNorm2d(nn.Module):
    """LayerNorm over the channel dim of a (B, C, H, W) tensor."""

    def __init__(self, channels, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(channels))
        self.bias = nn.Parameter(torch.zeros(channels))
        self.eps = eps

    def forward(self, x):
        mu = x.mean(1, keepdim=True)
        var = (x - mu).pow(2).mean(1, keepdim=True)
        x = (x - mu) / torch.sqrt(var + self.eps)
        return x * self.weight[None, :, None, None] + self.bias[None, :, None, None]


class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2


class NAFBlock(nn.Module):
    """One NAFNet block: simplified channel attention + SimpleGate,
    two residual sub-blocks, learnable per-channel residual scales."""

    def __init__(self, channels, expand=2, ffn_expand=2):
        super().__init__()
        dw_channels = channels * expand

        self.norm1 = LayerNorm2d(channels)
        self.conv1 = nn.Conv2d(channels, dw_channels, 1)
        self.dwconv = nn.Conv2d(dw_channels, dw_channels, 3, padding=1, groups=dw_channels)
        self.sg1 = SimpleGate()
        self.sca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(dw_channels // 2, dw_channels // 2, 1),
        )
        self.conv2 = nn.Conv2d(dw_channels // 2, channels, 1)
        self.beta = nn.Parameter(torch.zeros(1, channels, 1, 1))

        ffn_channels = channels * ffn_expand
        self.norm2 = LayerNorm2d(channels)
        self.conv3 = nn.Conv2d(channels, ffn_channels, 1)
        self.sg2 = SimpleGate()
        self.conv4 = nn.Conv2d(ffn_channels // 2, channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1, channels, 1, 1))

    def forward(self, x):
        y = self.norm1(x)
        y = self.conv1(y)
        y = self.dwconv(y)
        y = self.sg1(y)
        y = y * self.sca(y)
        y = self.conv2(y)
        x = x + y * self.beta

        y = self.norm2(x)
        y = self.conv3(y)
        y = self.sg2(y)
        y = self.conv4(y)
        x = x + y * self.gamma

        return x


class Down(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.op = nn.Conv2d(channels, channels * 2, 2, stride=2)

    def forward(self, x):
        return self.op(x)


class Up(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.op = nn.Sequential(
            nn.Conv2d(channels, channels * 2, 1, bias=False),
            nn.PixelShuffle(2),
        )

    def forward(self, x):
        return self.op(x)


class NAFNetLiteSR(nn.Module):
    """
    width: base channel count (32 keeps this fast on H100; bump to 48/64
           if there's inference-time headroom left after benchmarking).
    enc_blocks / dec_blocks: block counts per encoder/decoder stage.
    middle_blocks: block count at the bottleneck.
    """

    def __init__(self, in_ch=1, out_ch=1, width=32, enc_blocks=(2, 2), middle_blocks=4, dec_blocks=(2, 2)):
        super().__init__()
        self.intro = nn.Conv2d(in_ch, width, 3, padding=1)

        self.encoders = nn.ModuleList()
        self.downs = nn.ModuleList()
        ch = width
        for n in enc_blocks:
            self.encoders.append(nn.Sequential(*[NAFBlock(ch) for _ in range(n)]))
            self.downs.append(Down(ch))
            ch *= 2

        self.middle = nn.Sequential(*[NAFBlock(ch) for _ in range(middle_blocks)])

        self.ups = nn.ModuleList()
        self.decoders = nn.ModuleList()
        for n in dec_blocks:
            self.ups.append(Up(ch))
            ch //= 2
            self.decoders.append(nn.Sequential(*[NAFBlock(ch) for _ in range(n)]))

        # Fused 2x super-resolution head: width -> 4*width -> PixelShuffle(2) -> width, at 2x spatial res.
        self.sr_expand = nn.Conv2d(width, width * 4, 3, padding=1)
        self.sr_shuffle = nn.PixelShuffle(2)
        self.sr_refine = NAFBlock(width)
        self.out_conv = nn.Conv2d(width, out_ch, 3, padding=1)

        self.padder_size = 2 ** len(enc_blocks)

    def _pad_to_multiple(self, x):
        _, _, h, w = x.shape
        pad_h = (self.padder_size - h % self.padder_size) % self.padder_size
        pad_w = (self.padder_size - w % self.padder_size) % self.padder_size
        return F.pad(x, (0, pad_w, 0, pad_h), mode="reflect"), (h, w)

    def forward(self, x):
        x, (orig_h, orig_w) = self._pad_to_multiple(x)

        x = self.intro(x)
        skips = []
        for enc, down in zip(self.encoders, self.downs):
            x = enc(x)
            skips.append(x)
            x = down(x)

        x = self.middle(x)

        for up, dec, skip in zip(self.ups, self.decoders, reversed(skips)):
            x = up(x)
            x = x + skip
            x = dec(x)

        x = self.sr_expand(x)
        x = self.sr_shuffle(x)
        x = self.sr_refine(x)
        x = self.out_conv(x)

        # Crop back from any reflect-padding, then account for the 2x SR head.
        x = x[:, :, : orig_h * 2, : orig_w * 2]
        return x


def build_model():
    return NAFNetLiteSR(in_ch=1, out_ch=1, width=32, enc_blocks=(2, 2), middle_blocks=4, dec_blocks=(2, 2))


if __name__ == "__main__":
    m = build_model()
    x = torch.randn(1, 1, 128, 128)
    y = m(x)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"input {tuple(x.shape)} -> output {tuple(y.shape)}, params={n_params/1e6:.2f}M")


In [ ]:
%%writefile /content/src/losses.py
"""
Loss functions for training.

Deliberately no adversarial/GAN loss anywhere -- the problem statement
explicitly warns against "artificial patterns or ringing", which is the
classic failure mode of GAN-based restoration (Real-ESRGAN etc.).

Combined loss = Charbonnier (robust pixel fidelity)
              + SSIM         (structural similarity, matches a grading metric)
              + small-weight VGG perceptual (helps the LPIPS grading metric,
                kept low-weight so it can't dominate and hallucinate detail)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class CharbonnierLoss(nn.Module):
    """Robust L1 (smooth near zero) -- standard for restoration tasks,
    less prone to over-smoothing than plain L2/MSE."""

    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        diff = pred - target
        return torch.mean(torch.sqrt(diff * diff + self.eps * self.eps))


def _gaussian_window(window_size, sigma, device, dtype):
    coords = torch.arange(window_size, device=device, dtype=dtype) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window_2d = g[:, None] @ g[None, :]
    return window_2d[None, None, :, :]


class SSIMLoss(nn.Module):
    """1 - SSIM, computed with a Gaussian window. Single-channel only
    (matches the grayscale-only nature of this dataset)."""

    def __init__(self, window_size=11, sigma=1.5, data_range=1.0):
        super().__init__()
        self.window_size = window_size
        self.sigma = sigma
        self.data_range = data_range
        self.register_buffer("_window_cache", torch.empty(0), persistent=False)

    def _ssim_map(self, pred, target):
        window = _gaussian_window(self.window_size, self.sigma, pred.device, pred.dtype)
        pad = self.window_size // 2

        mu_p = F.conv2d(pred, window, padding=pad)
        mu_t = F.conv2d(target, window, padding=pad)

        mu_p2, mu_t2, mu_pt = mu_p * mu_p, mu_t * mu_t, mu_p * mu_t

        sigma_p2 = F.conv2d(pred * pred, window, padding=pad) - mu_p2
        sigma_t2 = F.conv2d(target * target, window, padding=pad) - mu_t2
        sigma_pt = F.conv2d(pred * target, window, padding=pad) - mu_pt

        c1 = (0.01 * self.data_range) ** 2
        c2 = (0.03 * self.data_range) ** 2

        ssim_map = ((2 * mu_pt + c1) * (2 * sigma_pt + c2)) / (
            (mu_p2 + mu_t2 + c1) * (sigma_p2 + sigma_t2 + c2)
        )
        return ssim_map

    def forward(self, pred, target):
        return 1.0 - self._ssim_map(pred, target).mean()


class VGGPerceptualLoss(nn.Module):
    """Optional. Replicates the single grayscale channel to 3 channels
    to run through an ImageNet-pretrained VGG16. Requires internet on
    first use (downloads pretrained weights) -- fine on a cloud GPU
    training environment, not needed at KLA's inference/eval time."""

    def __init__(self, layer_idx=16):
        super().__init__()
        from torchvision.models import vgg16, VGG16_Weights

        vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features[:layer_idx].eval()
        for p in vgg.parameters():
            p.requires_grad = False
        self.vgg = vgg
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def _prep(self, x):
        x = x.repeat(1, 3, 1, 1)
        return (x - self.mean) / self.std

    def forward(self, pred, target):
        f_pred = self.vgg(self._prep(pred))
        f_target = self.vgg(self._prep(target))
        return F.l1_loss(f_pred, f_target)


class CombinedLoss(nn.Module):
    def __init__(self, w_charbonnier=1.0, w_ssim=0.2, w_perceptual=0.05, use_perceptual=True):
        super().__init__()
        self.charbonnier = CharbonnierLoss()
        self.ssim = SSIMLoss()
        self.w_charbonnier = w_charbonnier
        self.w_ssim = w_ssim
        self.w_perceptual = w_perceptual

        self.perceptual = None
        if use_perceptual:
            try:
                self.perceptual = VGGPerceptualLoss()
            except Exception as e:
                print(f"[losses] Perceptual loss disabled (couldn't load VGG weights: {e})")
                self.perceptual = None

    def forward(self, pred, target):
        pred_c = pred.clamp(0, 1)
        target_c = target.clamp(0, 1)

        loss_c = self.charbonnier(pred, target)
        loss_s = self.ssim(pred_c, target_c)
        total = self.w_charbonnier * loss_c + self.w_ssim * loss_s

        logs = {"charbonnier": loss_c.item(), "ssim_loss": loss_s.item()}

        if self.perceptual is not None:
            loss_p = self.perceptual(pred_c, target_c)
            total = total + self.w_perceptual * loss_p
            logs["perceptual"] = loss_p.item()

        logs["total"] = total.item()
        return total, logs


In [ ]:
%%writefile /content/src/train.py
"""
Training script -- reproduces the model from scratch (satisfies KLA's
GitHub requirement #3: "Python script or notebook that reproduces your
training process from scratch").

Usage:
    python train.py --data_root C:/semicon/train --epochs 100 --batch_size 16 \
        --device cuda --out_dir C:/semicon/checkpoints

On Colab, point --out_dir straight at a mounted Drive folder so best.pt/
last.pt never live only in the ephemeral /content filesystem, e.g.:
    python train.py --data_root /content/train --out_dir \
        /content/drive/MyDrive/kla_ps01/checkpoints --device cuda --use_perceptual

Training also writes resume.pt (model + optimizer + scheduler + epoch +
best_psnr) to --out_dir after every epoch, so a dropped Colab runtime can
pick back up with --resume instead of restarting from scratch. resume.pt
is separate from best.pt/last.pt, which stay plain state_dicts because
inference.py (and KLA's benchmarking harness) load them as-is.

Run with --smoke_test for a tiny few-step run to sanity check the full
pipeline before committing to a long run on a real GPU.
"""

import argparse
import os
import time

import numpy as np
import torch
from torch.utils.data import DataLoader

from dataset import make_splits, SubsetRestorationDataset
from model import build_model
from losses import CombinedLoss


def psnr(pred, target, data_range=1.0):
    mse = torch.mean((pred - target) ** 2).item()
    if mse == 0:
        return float("inf")
    return 10 * np.log10((data_range ** 2) / mse)


def evaluate(model, loader, device):
    model.eval()
    psnr_sum, n = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            noisy = batch["noisy_lr"].to(device)
            gt = batch["gt"].to(device)
            pred = model(noisy).clamp(0, 1)
            for i in range(pred.shape[0]):
                psnr_sum += psnr(pred[i], gt[i])
                n += 1
    model.train()
    return psnr_sum / max(n, 1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_root", default="C:/semicon/train", help="folder containing GT/ and NoisyLR/")
    ap.add_argument("--out_dir", default="C:/semicon/checkpoints")
    ap.add_argument("--epochs", type=int, default=100)
    ap.add_argument("--batch_size", type=int, default=16)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--val_fraction", type=float, default=0.1)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--use_perceptual", action="store_true")
    ap.add_argument("--num_workers", type=int, default=4)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--smoke_test", action="store_true", help="run a handful of steps only, to sanity check the pipeline")
    ap.add_argument("--resume", default=None, help="path to a resume.pt checkpoint to continue training from")
    ap.add_argument("--ckpt_every", type=int, default=1, help="save resume.pt every N epochs")
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)

    os.makedirs(args.out_dir, exist_ok=True)
    gt_dir = os.path.join(args.data_root, "GT")
    noisy_dir = os.path.join(args.data_root, "NoisyLR")

    train_ids, val_ids = make_splits(gt_dir, noisy_dir, val_fraction=args.val_fraction, seed=args.seed)
    train_ds = SubsetRestorationDataset(gt_dir, noisy_dir, train_ids, train=True)
    val_ds = SubsetRestorationDataset(gt_dir, noisy_dir, val_ids, train=False)

    if args.smoke_test:
        train_ds.gt_paths = train_ds.gt_paths[:8]
        val_ds.gt_paths = val_ds.gt_paths[:4]
        args.epochs = 1
        args.num_workers = 0
        args.batch_size = min(args.batch_size, 4)

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,
                               num_workers=args.num_workers, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False,
                             num_workers=0)

    device = torch.device(args.device)
    model = build_model().to(device)
    loss_fn = CombinedLoss(use_perceptual=args.use_perceptual).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    print(f"train={len(train_ds)} val={len(val_ds)} device={device} "
          f"params={sum(p.numel() for p in model.parameters())/1e6:.2f}M")

    best_psnr = -1.0
    start_epoch = 0
    max_steps = 3 if args.smoke_test else None

    if args.resume:
        ckpt = torch.load(args.resume, map_location=device)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        best_psnr = ckpt["best_psnr"]
        start_epoch = ckpt["epoch"] + 1
        print(f"resumed from {args.resume} at epoch {start_epoch} (best_psnr={best_psnr:.2f}dB)")

    for epoch in range(start_epoch, args.epochs):
        t0 = time.time()
        running = {}
        step = 0
        for batch in train_loader:
            noisy = batch["noisy_lr"].to(device)
            gt = batch["gt"].to(device)

            pred = model(noisy)
            loss, logs = loss_fn(pred, gt)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            for k, v in logs.items():
                running[k] = running.get(k, 0.0) + v
            step += 1
            if max_steps and step >= max_steps:
                break

        scheduler.step()
        avg = {k: v / step for k, v in running.items()}
        val_psnr = evaluate(model, val_loader, device)
        dt = time.time() - t0

        print(f"epoch {epoch+1}/{args.epochs} | loss {avg['total']:.4f} "
              f"(charb {avg['charbonnier']:.4f} ssim {avg['ssim_loss']:.4f}) "
              f"| val_psnr {val_psnr:.2f}dB | {dt:.1f}s")

        if val_psnr > best_psnr:
            best_psnr = val_psnr
            torch.save(model.state_dict(), os.path.join(args.out_dir, "best.pt"))

        torch.save(model.state_dict(), os.path.join(args.out_dir, "last.pt"))

        if (epoch + 1) % args.ckpt_every == 0 or epoch == args.epochs - 1:
            torch.save({
                "epoch": epoch,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "best_psnr": best_psnr,
            }, os.path.join(args.out_dir, "resume.pt"))

    print(f"done. best val_psnr={best_psnr:.2f}dB, weights in {args.out_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/src/inference.py
"""
KLA-spec evaluation/inference script.

Per the problem statement, this file "will be used AS-IS by KLA's
benchmarking team ... on the H100 GPU. If your script does not run
without manual edits, your submission cannot be benchmarked."
So: no hardcoded absolute paths, no notebook-only state, sane defaults,
auto-creates the output directory, and works whether invoked with
positional args or named flags (the exact CLI contract KLA will use
wasn't specified beyond "accepts a path to the test images directory
and a path to the output directory").

Usage:
    python inference.py <input_dir> <output_dir>
    python inference.py --input_dir <input_dir> --output_dir <output_dir> --weights <path/to/best.pt>

Input format: .npy files, float32, 128x128 (or 256x256 for the 256->512
scale case), grayscale, matching the format KLA's own training data was
delivered in. .png/.tif/.jpg inputs are also accepted as a fallback.
Output: one .npy file per input, float32 in [0, 1], upscaled 2x, written
to <output_dir>/<same_filename>.npy -- same format as the GT files, so
it drops straight into their SSIM/PSNR/LPIPS scorer.
"""

import argparse
import os
import sys
import time

import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from model import build_model

DEFAULT_WEIGHTS = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "checkpoints", "best.pt")

IMAGE_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")


def load_input(path):
    if path.lower().endswith(".npy"):
        arr = np.load(path).astype(np.float32)
    elif path.lower().endswith(IMAGE_EXTS):
        from PIL import Image
        img = Image.open(path).convert("L")
        arr = np.asarray(img).astype(np.float32) / 255.0
    else:
        raise ValueError(f"Unsupported input file type: {path}")
    return arr


def save_output(arr, out_path):
    arr = np.clip(arr, 0.0, 1.0).astype(np.float32)
    np.save(out_path, arr)


def parse_args():
    ap = argparse.ArgumentParser(description="Run KLA restoration model inference on a directory of test images.")
    ap.add_argument("input_dir", nargs="?", default=None, help="path to test images directory")
    ap.add_argument("output_dir", nargs="?", default=None, help="path to write restored outputs")
    ap.add_argument("--input_dir", dest="input_dir_flag", default=None)
    ap.add_argument("--output_dir", dest="output_dir_flag", default=None)
    ap.add_argument("--weights", default=DEFAULT_WEIGHTS, help="path to trained model weights (.pt)")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = ap.parse_args()

    input_dir = args.input_dir_flag or args.input_dir
    output_dir = args.output_dir_flag or args.output_dir
    if not input_dir or not output_dir:
        ap.error("both an input directory and an output directory are required "
                  "(positionally or via --input_dir/--output_dir)")
    return input_dir, output_dir, args.weights, args.device


def main():
    input_dir, output_dir, weights_path, device_str = parse_args()

    if not os.path.isdir(input_dir):
        raise NotADirectoryError(f"input_dir does not exist: {input_dir}")
    os.makedirs(output_dir, exist_ok=True)

    device = torch.device(device_str)
    model = build_model().to(device)
    state = torch.load(weights_path, map_location=device)
    model.load_state_dict(state)
    model.eval()

    files = sorted(
        f for f in os.listdir(input_dir)
        if f.lower().endswith(".npy") or f.lower().endswith(IMAGE_EXTS)
    )
    if not files:
        print(f"[inference] WARNING: no .npy/image files found in {input_dir}")
        return

    print(f"[inference] {len(files)} files | device={device} | weights={weights_path}")

    times = []
    with torch.no_grad():
        for fname in files:
            in_path = os.path.join(input_dir, fname)
            arr = load_input(in_path)

            x = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).to(device)  # 1x1xHxW

            t0 = time.time()
            pred = model(x)
            if device.type == "cuda":
                torch.cuda.synchronize()
            dt = time.time() - t0
            times.append(dt)

            pred_np = pred.squeeze(0).squeeze(0).cpu().numpy()
            out_name = os.path.splitext(fname)[0] + ".npy"
            save_output(pred_np, os.path.join(output_dir, out_name))

    avg_ms = 1000 * sum(times) / len(times)
    print(f"[inference] done. {len(files)} images -> {output_dir}")
    print(f"[inference] avg inference time: {avg_ms:.1f} ms/image "
          f"(min {1000*min(times):.1f} ms, max {1000*max(times):.1f} ms)")


if __name__ == "__main__":
    main()


## 6. Sanity check: smoke test on GPU (a few steps only)

In [ ]:
!cd /content/src && python train.py --smoke_test --device cuda --data_root "{DATA_ROOT}"


## 7. Full training run

Adjust `--epochs` / `--batch_size` based on how much Colab session time
you have. `--out_dir` points straight at Drive, so `best.pt`/`last.pt`/
`resume.pt` are saved to Drive after every epoch -- no separate copy step,
and a Colab disconnect can't lose anything beyond the last epoch.

If a run gets interrupted, re-run this same cell after uncommenting the
`--resume` line below to continue from the last saved epoch instead of
starting over.

In [ ]:
CKPT_DIR = "/content/drive/MyDrive/kla_ps01/checkpoints"
!mkdir -p "{CKPT_DIR}"
!cd /content/src && python train.py     --data_root "{DATA_ROOT}"     --out_dir "{CKPT_DIR}"     --epochs 100     --batch_size 16     --lr 2e-4     --device cuda     --use_perceptual
    # --resume "{CKPT_DIR}/resume.pt"   # <-- uncomment to continue after a disconnect


## 8. Resume after a disconnect

No longer needed for a normal run -- checkpoints already live on Drive
(step 7). Use this only as a manual fallback: it re-runs training from
the last saved `resume.pt`, picking up epoch/optimizer/LR-schedule state
where it left off.

In [ ]:
CKPT_DIR = "/content/drive/MyDrive/kla_ps01/checkpoints"
!cd /content/src && python train.py     --data_root "{DATA_ROOT}"     --out_dir "{CKPT_DIR}"     --epochs 100     --batch_size 16     --lr 2e-4     --device cuda     --use_perceptual     --resume "{CKPT_DIR}/resume.pt"


## 9. Verify a checkpoint loads cleanly

Quick disconnect-recovery sanity check -- confirms a saved checkpoint is
intact and loadable before you rely on it.

In [ ]:
import torch, sys
sys.path.insert(0, "/content/src")
from model import build_model
m = build_model()
state = torch.load("/content/drive/MyDrive/kla_ps01/checkpoints/best.pt", map_location="cpu")
m.load_state_dict(state)
print("Checkpoint loads cleanly. Params:", sum(p.numel() for p in m.parameters()))


## 10. Run inference on the held-out test set + benchmark speed

Point `TEST_INPUT_DIR` at your uploaded held-out `NoisyLR/`-style test
folder (no GT). This mirrors exactly what KLA will run against your repo,
and reports the same avg ms/image figure you'll want for Slide 7 of the
PPT.

In [ ]:
TEST_INPUT_DIR = "/content/drive/MyDrive/kla_ps01/NoisyLR"  # <-- EDIT THIS
TEST_OUTPUT_DIR = "/content/outputs"

!cd /content/src && python inference.py "{TEST_INPUT_DIR}" "{TEST_OUTPUT_DIR}" \
    --weights /content/drive/MyDrive/kla_ps01/checkpoints/best.pt --device cuda


## 11. Compute SSIM / PSNR / LPIPS on your own validation split

(For final Slide 6 numbers. Requires ground truth, so this runs against
your local val split, not the blind test set.)

In [ ]:
import torch, numpy as np, lpips, sys
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
sys.path.insert(0, "/content/src")
from dataset import make_splits, SubsetRestorationDataset
from model import build_model
from torch.utils.data import DataLoader

device = "cuda"
gt_dir = DATA_ROOT + "/GT"
noisy_dir = DATA_ROOT + "/NoisyLR"
_, val_ids = make_splits(gt_dir, noisy_dir, val_fraction=0.1, seed=42)
val_ds = SubsetRestorationDataset(gt_dir, noisy_dir, val_ids, train=False)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

model = build_model().to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/kla_ps01/checkpoints/best.pt", map_location=device))
model.eval()

lpips_fn = lpips.LPIPS(net='alex').to(device)

ssims, psnrs, lpipss = [], [], []
with torch.no_grad():
    for batch in val_loader:
        noisy = batch["noisy_lr"].to(device)
        gt = batch["gt"].to(device)
        pred = model(noisy).clamp(0, 1)

        lp = lpips_fn(pred * 2 - 1, gt * 2 - 1).squeeze().cpu().numpy()
        lpipss.extend(np.atleast_1d(lp).tolist())

        pred_np = pred.squeeze(1).cpu().numpy()
        gt_np = gt.squeeze(1).cpu().numpy()
        for p, g in zip(pred_np, gt_np):
            ssims.append(ssim_fn(g, p, data_range=1.0))
            psnrs.append(psnr_fn(g, p, data_range=1.0))

print(f"SSIM:  {np.mean(ssims):.4f}")
print(f"PSNR:  {np.mean(psnrs):.2f} dB")
print(f"LPIPS: {np.mean(lpipss):.4f}")
